# Model evaluation visuals

This notebook creates clear, report-ready charts from the champion model saved by notebook 02. Run **01_data_cleaning.ipynb** and **02_model_training.ipynb** first.

The train/test log-loss comparison checks generalisation. ROC-AUC, PR-AUC, Brier score, and calibration remain the primary decision-quality measures.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    brier_score_loss,
    log_loss,
)

sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)


## Load the saved champion

The champion is selected by validation PR-AUC in notebook 02. The test set has not been used to select it.

In [ ]:
bundle = joblib.load(ROOT / 'artifacts' / 'benchmark_models.pkl')
results = bundle['results'].sort_values(
    'validation_pr_auc_mean',
    ascending=False,
).reset_index(drop=True)

champion_name = results.loc[0, 'model']
model = bundle['models'][champion_name]
X_test = bundle['X_test']
y_test = bundle['y_test']

clean = pd.read_csv(ROOT / 'data' / 'telco_clean_32col.csv')
X_train = clean.loc[~clean.index.isin(X_test.index)].drop(columns='Churn')
y_train = clean.loc[~clean.index.isin(X_test.index), 'Churn']

train_probability = model.predict_proba(X_train)[:, 1]
test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.50).astype(int)

print(f'Champion model: {champion_name}')
display(results.head())


## 1. Dataset quality and feature relationships

These Seaborn charts show the class imbalance and the numerical features most associated with the churn label. Correlation is descriptive only; it is not proof of causation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=clean, x='Churn', hue='Churn', legend=False, ax=axes[0])
axes[0].set_title('Churn class balance')
axes[0].set_xticklabels(['Stayed', 'Churned'])
axes[0].set_xlabel('Observed outcome')

correlations = clean.corr(numeric_only=True)['Churn'].drop('Churn').abs().nlargest(12).index
heatmap_data = clean[list(correlations) + ['Churn']].corr()
sns.heatmap(heatmap_data, cmap='vlag', center=0, annot=False, ax=axes[1])
axes[1].set_title('Top numerical correlations with churn')

plt.tight_layout()
plt.savefig(FIGURES / 'data_quality_and_correlation.png', dpi=160, bbox_inches='tight')
plt.show()


## 2. Train versus test loss

Log loss rewards well-calibrated probabilities. A small train/test gap is healthier than a large gap; a lower value is better. This is a true held-out test calculation, not training accuracy.

In [ ]:
loss_summary = pd.DataFrame({
    'split': ['Training', 'Held-out test'],
    'log_loss': [
        log_loss(y_train, train_probability),
        log_loss(y_test, test_probability),
    ],
})

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=loss_summary, x='split', y='log_loss', hue='split', legend=False, ax=ax)
ax.bar_label(ax.containers[0], fmt='%.3f', padding=3)
ax.set_title(f'Train and test log loss: {champion_name}')
ax.set_xlabel('')
ax.set_ylabel('Log loss (lower is better)')
plt.tight_layout()
plt.savefig(FIGURES / 'train_test_log_loss.png', dpi=160, bbox_inches='tight')
plt.show()

display(loss_summary)


## 3. Classification and ranking behaviour

The confusion matrix uses the default 0.50 threshold only for reference. The retention simulation selects a threshold from campaign value and budget instead.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_prediction,
    display_labels=['Stayed', 'Churned'],
    cmap='Blues',
    ax=axes[0],
)
axes[0].set_title('Confusion matrix at threshold 0.50')

RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[1], color='#1f77b4')
axes[1].set_title('ROC curve')

PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[2], color='#d62728')
axes[2].set_title('Precision-recall curve')

plt.tight_layout()
plt.savefig(FIGURES / 'classification_and_ranking_curves.png', dpi=160, bbox_inches='tight')
plt.show()


## 4. Probability distribution and calibration

A calibrated 0.70 prediction should correspond to roughly 70% churn in a comparable group. This chart is essential before using probabilities in a retention budget.

In [ ]:
observed_rate, mean_probability = calibration_curve(
    y_test,
    test_probability,
    n_bins=10,
    strategy='quantile',
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=pd.DataFrame({'Churn outcome': y_test, 'Predicted probability': test_probability}),
    x='Predicted probability',
    hue='Churn outcome',
    bins=20,
    stat='density',
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title('Predicted churn probability by outcome')

axes[1].plot([0, 1], [0, 1], '--', color='black', label='Perfect calibration')
axes[1].plot(mean_probability, observed_rate, marker='o', label=champion_name)
axes[1].set_title(f'Calibration curve | Brier = {brier_score_loss(y_test, test_probability):.3f}')
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Observed churn rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'probability_distribution_and_calibration.png', dpi=160, bbox_inches='tight')
plt.show()


## Report sentence

Use the train/test log-loss chart to discuss generalisation, the ROC/PR charts to discuss ranking, and the calibration curve plus Brier score to justify using probabilities in the retention simulation.